# Rizzvision v5 — Dataset Curation

Add these as input datasets before running:
- `nitin2807/deepfashion2-full`
- `nitin2807/fashionpedia-full`
- `nitin2807/imaterialist-fashion-2020`
- `aryashah2k/large-shoe-dataset-ut-zappos50k`
- `noobyogi0100/shoe-dataset`
- `hasibalmuzdadid/shoe-vs-sandal-vs-boot-dataset-15k-images`
- `iaasileperuma/footwear-images-dataset`
- `kritanjalijain/outfititems`
- `paramaggarwal/fashion-product-images-dataset`
- `agrigorev/clothing-dataset-full`
- `silverstone1903/deep-fashion-multimodal`

In [ ]:
import os, json, csv, random, shutil
import concurrent.futures
from pathlib import Path
from collections import defaultdict
import pandas as pd
from tqdm.notebook import tqdm

BASE       = '/kaggle/input/datasets/nitin2807'
DF2_DIR    = f'{BASE}/deepfashion2-full'
FP_DIR     = f'{BASE}/fashionpedia-full'
IMAT_DIR   = f'{BASE}/imaterialist-fashion-2020'

# Footwear datasets
ZAPPOS_DIR   = '/kaggle/input/datasets/aryashah2k/large-shoe-dataset-ut-zappos50k'
SHOE_DS_DIR  = '/kaggle/input/datasets/noobyogi0100/shoe-dataset'
SVB_DIR      = '/kaggle/input/datasets/hasibalmuzdadid/shoe-vs-sandal-vs-boot-dataset-15k-images'
FW_IMG_DIR   = '/kaggle/input/datasets/iaasileperuma/footwear-images-dataset'

# General clothing datasets
OUTFIT_DIR   = '/kaggle/input/datasets/kritanjalijain/outfititems'
FPI_DIR      = '/kaggle/input/datasets/paramaggarwal/fashion-product-images-dataset'
CLOTHING_DIR = '/kaggle/input/datasets/agrigorev/clothing-dataset-full'
DFMM_DIR     = '/kaggle/input/datasets/silverstone1903/deep-fashion-multimodal'

OUT_DIR    = '/kaggle/working'

LABELS        = ['tops', 'bottoms', 'footwear', 'outerwear', 'dress']
MAX_PER_CLASS = 40000   # v5: raised from 25K
VAL_FRAC      = 0.10
TEST_FRAC     = 0.10
SEED          = 42
NUM_WORKERS   = 8

os.makedirs(f'{OUT_DIR}/images/train', exist_ok=True)
os.makedirs(f'{OUT_DIR}/images/val',   exist_ok=True)
os.makedirs(f'{OUT_DIR}/images/test',  exist_ok=True)

# Dataset availability check
datasets = {
    'DF2':      DF2_DIR,    'FP':       FP_DIR,     'iMat':     IMAT_DIR,
    'Zappos':   ZAPPOS_DIR, 'ShoeDS':   SHOE_DS_DIR, 'SvB':     SVB_DIR,
    'FwImg':    FW_IMG_DIR, 'Outfit':   OUTFIT_DIR,
    'FPI':      FPI_DIR,    'Clothing': CLOTHING_DIR, 'DFMM':   DFMM_DIR,
}
for name, d in datasets.items():
    if Path(d).exists():
        items = os.listdir(d)
        print(f'[OK] {name:<10}: {items[:6]}')
    else:
        print(f'[MISSING] {name:<10}: {d}')
print('Config ready')

In [ ]:
DF2_CAT_TO_LABEL = {
    1:'tops', 2:'tops', 5:'tops', 6:'tops',
    3:'outerwear', 4:'outerwear',
    7:'bottoms', 8:'bottoms', 9:'bottoms',
    10:'dress', 11:'dress', 12:'dress', 13:'dress',
}

# Supercategory map — covers Fashionpedia and iMaterialist actual keys
# (confirmed from annotation files: supercategories are 'upperbody', 'lowerbody',
#  'wholebody', 'outerwear', 'legs and feet' — NOT 'feet' or 'footwear')
SUPER_TO_LABEL = {
    'upperbody':    'tops',      'Upper-body':   'tops',
    'lowerbody':    'bottoms',   'Lower-body':   'bottoms',
    'wholebody':    'dress',     'Whole-body':   'dress',
    'outerwear':    'outerwear', 'Outerwear':    'outerwear',
    'legs and feet':'footwear',  'Legs and feet':'footwear',  # actual FP/iMat key
    'feet':         'footwear',  'Feet':         'footwear',  # fallback variants
    'footwear':     'footwear',  'Footwear':     'footwear',
}

OI_FOOTWEAR = {'/m/0fly7','/m/01b638','/m/03yfpv','/m/02gzp','/m/0_cp5'}

def make_row(filepath, label_set):
    row = {'filepath': str(filepath)}
    for l in LABELS:
        row[l] = 1 if l in label_set else 0
    return row

footwear_sources = {}

print('Mappings ready')
print(f'SUPER_TO_LABEL keys: {list(SUPER_TO_LABEL.keys())}')

In [ ]:
# ── DeepFashion2 (parallel) ────────────────────────────────────────────────
records = []

def process_df2_anno(af, image_dir):
    try: data = json.loads(af.read_text())
    except: return None
    if not isinstance(data, dict): return None  # skip malformed files
    label_set = set()
    for k, v in data.items():
        if k.startswith('item') and isinstance(v, dict):
            lbl = DF2_CAT_TO_LABEL.get(v.get('category_id'))
            if lbl: label_set.add(lbl)
    if not label_set: return None
    src = next((image_dir / f'{af.stem}{ext}' for ext in ('.jpg','.JPG','.png')
                if (image_dir / f'{af.stem}{ext}').exists()), None)
    return make_row(src, label_set) if src else None

for split, anno_subdir, img_subdir in [
    ('train',      'train',               'train'),
    ('validation', 'json_for_validation', 'validation'),
]:
    base = Path(DF2_DIR) / anno_subdir
    anno_dir  = base / 'annos' if (base / 'annos').exists() else base
    image_dir = Path(DF2_DIR) / img_subdir / 'image'
    if not image_dir.exists(): image_dir = Path(DF2_DIR) / img_subdir
    anno_files = sorted(anno_dir.glob('*.json')) if anno_dir.exists() else []
    if not anno_files:
        print(f'[DF2] {split}: no annotation files found at {anno_dir}'); continue
    print(f'[DF2] {split}: {len(anno_files):,} files, images at {image_dir}')
    with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
        futs = {ex.submit(process_df2_anno, af, image_dir): af for af in anno_files}
        for fut in tqdm(concurrent.futures.as_completed(futs), total=len(futs), desc=f'DF2/{split}'):
            result = fut.result()
            if result: records.append(result)

print(f'After DF2: {len(records):,} records')

In [ ]:
# ── Fashionpedia (parallel) ────────────────────────────────────────────────
fp_anno_dir = Path(FP_DIR) / 'annotations'
fp_img_root = Path(FP_DIR) / 'images'
fp_img_dirs = [fp_img_root] + ([p for p in fp_img_root.iterdir() if p.is_dir()] if fp_img_root.exists() else [])

def process_fp_item(img_id, label_set, id_to_file, img_dirs):
    fname = id_to_file.get(img_id)
    if not fname: return None
    src = next((d / fname for d in img_dirs if (d / fname).exists()), None)
    return make_row(src, label_set) if src else None

fp_footwear = 0
for anno_file in ('instances_attributes_train2020.json', 'instances_attributes_val2020.json'):
    anno_path = fp_anno_dir / anno_file
    if not anno_path.exists():
        print(f'[FP] {anno_file} not found, skipping'); continue
    data = json.loads(anno_path.read_text())

    # Print actual supercategories present so we can verify mapping coverage
    all_supers = sorted(set(cat.get('supercategory','') for cat in data.get('categories', [])))
    print(f'[FP] {anno_file} supercategories: {all_supers}')

    cat_map = {cat['id']: SUPER_TO_LABEL[cat['supercategory']]
               for cat in data.get('categories', [])
               if cat.get('supercategory') in SUPER_TO_LABEL}
    img_labels = defaultdict(set)
    for ann in data.get('annotations', []):
        lbl = cat_map.get(ann.get('category_id'))
        if lbl: img_labels[ann['image_id']].add(lbl)
    id_to_file = {img['id']: img['file_name'] for img in data.get('images', [])}

    fw_count = sum(1 for ls in img_labels.values() if 'footwear' in ls)
    print(f'[FP] {anno_file}: {len(img_labels):,} labelled images  |  footwear images: {fw_count:,}')
    fp_footwear += fw_count

    with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
        futs = [ex.submit(process_fp_item, img_id, label_set, id_to_file, fp_img_dirs)
                for img_id, label_set in img_labels.items()]
        for fut in tqdm(concurrent.futures.as_completed(futs), total=len(futs), desc='FP'):
            result = fut.result()
            if result: records.append(result)

footwear_sources['fashionpedia'] = fp_footwear
print(f'After Fashionpedia: {len(records):,} records  |  FP footwear: {fp_footwear:,}')

In [ ]:
# ── iMaterialist 2020 (parallel) ───────────────────────────────────────────
csv.field_size_limit(10 * 1024 * 1024)

label_desc_path = Path(IMAT_DIR) / 'label_descriptions.json'
train_csv_path  = Path(IMAT_DIR) / 'train.csv'
train_imgs_path = Path(IMAT_DIR) / 'train'

desc = json.loads(label_desc_path.read_text())

# Print actual supercategories present in iMat
all_supers = sorted(set(cat.get('supercategory','') for cat in desc.get('categories', [])))
print(f'[iMat] supercategories: {all_supers}')

cat_map = {int(cat['id']): SUPER_TO_LABEL[cat['supercategory']]
           for cat in desc.get('categories', [])
           if cat.get('supercategory') in SUPER_TO_LABEL}
print(f'[iMat] {len(cat_map)} mapped label IDs')

img_labels = defaultdict(set)
with open(train_csv_path) as f:
    for row in csv.DictReader(f):
        for cid_str in row.get('ClassId','').replace(',',' ').split():
            try:
                lbl = cat_map.get(int(cid_str.split('_')[0]))
                if lbl: img_labels[row['ImageId']].add(lbl)
            except ValueError: pass

imat_footwear = sum(1 for ls in img_labels.values() if 'footwear' in ls)
print(f'[iMat] {len(img_labels):,} labelled images  |  footwear images: {imat_footwear:,}')

def process_imat_item(img_id, label_set, imgs_path):
    src = next((imgs_path / f'{img_id}{ext}' for ext in ('.jpg','.png')
                if (imgs_path / f'{img_id}{ext}').exists()), None)
    return make_row(src, label_set) if src else None

with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
    futs = [ex.submit(process_imat_item, img_id, label_set, train_imgs_path)
            for img_id, label_set in img_labels.items()]
    for fut in tqdm(concurrent.futures.as_completed(futs), total=len(futs), desc='iMat'):
        result = fut.result()
        if result: records.append(result)

footwear_sources['imat'] = imat_footwear
print(f'After iMat: {len(records):,} records  |  iMat footwear: {imat_footwear:,}')

In [ ]:
# ── Footwear: all shoe datasets ────────────────────────────────────────────────
# All four datasets are treated as footwear. We rglob all images regardless of
# subfolder structure (categories like shoe/sandal/boot all map to 'footwear').
# Cap total at MAX_PER_CLASS to keep balance.

FOOTWEAR_DIRS = {
    'zappos':  ZAPPOS_DIR,
    'shoe_ds': SHOE_DS_DIR,
    'svb':     SVB_DIR,
    'fw_img':  FW_IMG_DIR,
}
FOOTWEAR_CAP = MAX_PER_CLASS  # 25K total across all sources

def _collect_images(d):
    p = Path(d)
    if not p.exists():
        return []
    imgs = []
    for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG'):
        imgs.extend(p.rglob(ext))
    return imgs

all_fw_imgs = []
for src_name, src_dir in FOOTWEAR_DIRS.items():
    imgs = _collect_images(src_dir)
    print(f'[Footwear/{src_name}] {len(imgs):,} images found')
    footwear_sources[src_name] = len(imgs)
    all_fw_imgs.extend(imgs)

print(f'\nTotal footwear images across all sources: {len(all_fw_imgs):,}')

# Shuffle then cap
rng_fw = random.Random(SEED)
rng_fw.shuffle(all_fw_imgs)
all_fw_imgs = all_fw_imgs[:FOOTWEAR_CAP]
print(f'Using {len(all_fw_imgs):,} after cap ({FOOTWEAR_CAP:,})')

def add_footwear(src):
    return make_row(src, {'footwear'}) if Path(src).exists() else None

with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
    futs = [ex.submit(add_footwear, src) for src in all_fw_imgs]
    for fut in tqdm(concurrent.futures.as_completed(futs), total=len(futs), desc='Footwear'):
        result = fut.result()
        if result: records.append(result)

print(f'\nFootwear source breakdown:')
for src, count in footwear_sources.items():
    print(f'  {src:<15}: {count:,}')
print(f'\nAfter footwear: {len(records):,} records')

In [ ]:
# ── Outfit Items (general clothing — auto-map subfolders to labels) ────────────
OUTFIT_LABEL_MAP = {
    # tops
    'top': 'tops', 'tops': 'tops', 'tshirt': 'tops', 't-shirt': 'tops',
    'shirt': 'tops', 'blouse': 'tops', 'hoodie': 'tops', 'sweater': 'tops',
    'tank': 'tops', 'vest': 'tops', 'polo': 'tops', 'crop top': 'tops',
    'croptop': 'tops', 'tunic': 'tops', 'sweatshirt': 'tops',
    'upperwear': 'tops', 'upper': 'tops',
    # bottoms
    'bottom': 'bottoms', 'bottoms': 'bottoms', 'bottomwear': 'bottoms',
    'jeans': 'bottoms', 'pants': 'bottoms', 'trousers': 'bottoms',
    'shorts': 'bottoms', 'skirt': 'bottoms', 'leggings': 'bottoms',
    'joggers': 'bottoms', 'chinos': 'bottoms', 'culottes': 'bottoms',
    # outerwear
    'jacket': 'outerwear', 'coat': 'outerwear', 'outerwear': 'outerwear',
    'blazer': 'outerwear', 'cardigan': 'outerwear', 'parka': 'outerwear',
    'windbreaker': 'outerwear', 'trench': 'outerwear', 'bomber': 'outerwear',
    # dress
    'dress': 'dress', 'dresses': 'dress', 'jumpsuit': 'dress',
    'romper': 'dress', 'gown': 'dress', 'maxi': 'dress',
    'one-piece': 'dress', 'onepiece': 'dress',
    # footwear
    'shoe': 'footwear', 'shoes': 'footwear',
    'boot': 'footwear', 'boots': 'footwear',
    'sandal': 'footwear', 'sandals': 'footwear',
    'footwear': 'footwear', 'sneaker': 'footwear', 'sneakers': 'footwear',
    'heel': 'footwear', 'heels': 'footwear', 'high heel': 'footwear',
    'highheels': 'footwear', 'stiletto': 'footwear', 'stilettos': 'footwear',
    'flat': 'footwear', 'flats': 'footwear', 'loafer': 'footwear',
    'loafers': 'footwear', 'slipper': 'footwear', 'slippers': 'footwear',
    'flip flop': 'footwear', 'flipflop': 'footwear', 'flipflops': 'footwear',
    'flip-flop': 'footwear', 'flip-flops': 'footwear',
    'mule': 'footwear', 'mules': 'footwear', 'clog': 'footwear',
    'clogs': 'footwear', 'oxford': 'footwear', 'oxfords': 'footwear',
    'pump': 'footwear', 'pumps': 'footwear', 'wedge': 'footwear',
    'wedges': 'footwear', 'trainer': 'footwear', 'trainers': 'footwear',
    'espadrille': 'footwear', 'espadrilles': 'footwear',
    'ankle boot': 'footwear', 'ankleboot': 'footwear',
    'knee boot': 'footwear', 'kneeboot': 'footwear',
}

outfit_path = Path(OUTFIT_DIR)
outfit_added = 0

if not outfit_path.exists():
    print('[Outfit] dataset not found — skipping')
else:
    subfolders = [p for p in outfit_path.rglob('*') if p.is_dir()]
    print(f'[Outfit] discovered {len(subfolders)} subfolders:')
    for sf in sorted(subfolders)[:50]:
        key = sf.name.lower().strip()
        lbl = OUTFIT_LABEL_MAP.get(key)
        print(f'  {sf.name:<30} → {lbl or "UNMAPPED (skipped)"}')
    if len(subfolders) > 50:
        print(f'  ... and {len(subfolders)-50} more')

    for sf in subfolders:
        key = sf.name.lower().strip()
        lbl = OUTFIT_LABEL_MAP.get(key)
        if not lbl:
            continue
        imgs = list(sf.glob('*.jpg')) + list(sf.glob('*.jpeg')) + list(sf.glob('*.png'))
        for img in imgs:
            records.append(make_row(img, {lbl}))
            outfit_added += 1

    print(f'\n[Outfit] added {outfit_added:,} records')

print(f'After outfit items: {len(records):,} records')

In [ ]:
# ── Fashion Product Images (paramaggarwal) ────────────────────────────────────
# 44K products with masterCategory + subCategory + articleType labels.
# subCategory: Topwear, Bottomwear, Dress, Saree, Jumpsuit, Innerwear, Socks
# articleType: Jackets, Blazers, Coats, Windcheater, Trench Coat → outerwear
# masterCategory: Footwear → footwear

FPI_OUTERWEAR_TYPES = {
    'jackets', 'blazers', 'coats', 'windcheater', 'trench coat',
    'parka', 'bomber', 'overcoat', 'raincoat', 'shrug', 'cape',
}
FPI_DRESS_SUBCATS = {'dress', 'saree', 'jumpsuit', 'rompers', 'dungarees', 'skirts'}
# Note: skirts → dress is debatable but better than dropping them

fpi_path = Path(FPI_DIR)
fpi_added = 0

if not fpi_path.exists():
    print('[FPI] dataset not found — skipping')
else:
    # Styles CSV lives at root or in a subfolder
    styles_csv = next(fpi_path.rglob('styles.csv'), None)
    img_dir    = next((p for p in [fpi_path / 'images', fpi_path / 'Images'] if p.exists()), None)
    if not img_dir:
        # Some versions nest images under fashion-dataset/
        img_dir = next(fpi_path.rglob('images'), None)

    if styles_csv is None or img_dir is None:
        print(f'[FPI] Could not find styles.csv or images dir under {fpi_path}')
        print(f'  Contents: {os.listdir(fpi_path)[:10]}')
    else:
        print(f'[FPI] styles.csv: {styles_csv}')
        print(f'[FPI] images dir: {img_dir}')
        df_fpi = pd.read_csv(styles_csv, on_bad_lines='skip')
        print(f'[FPI] {len(df_fpi):,} rows  |  columns: {list(df_fpi.columns)}')

        for _, row in df_fpi.iterrows():
            master  = str(row.get('masterCategory', '')).lower().strip()
            subcat  = str(row.get('subCategory',    '')).lower().strip()
            article = str(row.get('articleType',    '')).lower().strip()
            img_id  = str(row.get('id', '')).strip()

            src = next((img_dir / f'{img_id}{ext}' for ext in ('.jpg', '.png', '.JPG')
                        if (img_dir / f'{img_id}{ext}').exists()), None)
            if src is None:
                continue

            label_set = set()
            if master == 'footwear':
                label_set.add('footwear')
            elif article in FPI_OUTERWEAR_TYPES:
                label_set.add('outerwear')
            elif subcat == 'topwear':
                label_set.add('tops')
            elif subcat == 'bottomwear':
                label_set.add('bottoms')
            elif subcat in FPI_DRESS_SUBCATS:
                label_set.add('dress')
            # Skip accessories, bags, watches, etc.

            if label_set:
                records.append(make_row(src, label_set))
                fpi_added += 1

        print(f'[FPI] added {fpi_added:,} records')

print(f'After FPI: {len(records):,} records')

In [ ]:
# ── Clothing Dataset Full (agrigorev) ─────────────────────────────────────────
# Structure (confirmed): flat images in images_original/ + images.csv with
# columns: image (filename), label (category string).
# Labels: dress, hat, longsleeve, outwear, pants, shirt, shoes, shorts, skirt, t-shirt

CLOTHING_LABEL_MAP = {
    't-shirt':    'tops',      'shirt':      'tops',
    'longsleeve': 'tops',      'blouse':     'tops',
    'top':        'tops',      'hoodie':     'tops',
    'pants':      'bottoms',   'shorts':     'bottoms',
    'skirt':      'bottoms',   'jeans':      'bottoms',
    'trousers':   'bottoms',
    'outwear':    'outerwear', 'jacket':     'outerwear',
    'coat':       'outerwear', 'blazer':     'outerwear',
    'dress':      'dress',
    'shoes':      'footwear',  'boots':      'footwear',
    'sandals':    'footwear',  'sneakers':   'footwear',
}

clothing_path = Path(CLOTHING_DIR)
clothing_added = 0

if not clothing_path.exists():
    print('[Clothing] dataset not found — skipping')
else:
    # Probe structure
    print(f'[Clothing] root contents: {os.listdir(clothing_path)[:10]}')

    csv_path = next(
        (clothing_path / f for f in ('images.csv',) if (clothing_path / f).exists()),
        next(clothing_path.rglob('images.csv'), None)
    )
    img_dir = next(
        (p for p in [clothing_path / 'images_original', clothing_path / 'images_compressed']
         if p.exists()),
        None
    )

    if csv_path is None or img_dir is None:
        print(f'[Clothing] Could not find images.csv or image dir — skipping')
        print(f'  csv_path={csv_path}  img_dir={img_dir}')
    else:
        print(f'[Clothing] CSV:    {csv_path}')
        print(f'[Clothing] images: {img_dir}')

        df_clothing = pd.read_csv(csv_path)
        print(f'[Clothing] {len(df_clothing):,} rows  |  columns: {list(df_clothing.columns)}')

        # Show label distribution
        label_col = 'label' if 'label' in df_clothing.columns else df_clothing.columns[-1]
        img_col   = 'image' if 'image' in df_clothing.columns else df_clothing.columns[0]
        print(f'[Clothing] using img_col={img_col!r}  label_col={label_col!r}')
        lbl_counts = df_clothing[label_col].value_counts()
        for lbl, cnt in lbl_counts.items():
            mapped = CLOTHING_LABEL_MAP.get(str(lbl).lower().strip())
            print(f'  {str(lbl):<20} {cnt:>6}  →  {mapped or "UNMAPPED (skipped)"}')

        clothing_skipped = 0
        for _, row in df_clothing.iterrows():
            lbl_raw = str(row.get(label_col, '')).lower().strip()
            lbl     = CLOTHING_LABEL_MAP.get(lbl_raw)
            if lbl is None:
                clothing_skipped += 1
                continue

            fname = str(row.get(img_col, '')).strip()
            if not fname:
                continue

            src = img_dir / fname
            if not src.exists():
                src = img_dir / (Path(fname).stem + '.jpg')
            if not src.exists():
                continue

            records.append(make_row(src, {lbl}))
            clothing_added += 1

        print(f'[Clothing] added {clothing_added:,} records  |  skipped {clothing_skipped:,}')

print(f'After Clothing dataset: {len(records):,} records')

In [ ]:
# ── DeepFashion-MultiModal (silverstone1903) ───────────────────────────────────
# 12,278 front-view full-body images of real people. Hero product labelled via
# product_type column in labels_front.csv. Images in selected_images/ subfolder.
#
# All 17 product_types confirmed from first run — zero unmapped expected.

DFMM_PRODUCT_MAP = {
    # tops
    'tees_tanks':          'tops',
    'blouses_shirts':      'tops',
    'sweaters':            'tops',
    'sweatshirts_hoodies': 'tops',
    'graphic_tees':        'tops',
    'shirts_polos':        'tops',   # confirmed unmapped on first run → tops
    'knits_tees':          'tops',
    'shirts_tops':         'tops',
    # bottoms
    'denim':               'bottoms',
    'pants':               'bottoms',
    'shorts':              'bottoms',
    'skirts':              'bottoms',
    'leggings':            'bottoms',
    'joggers_sweatpants':  'bottoms',
    # outerwear
    'jackets_vests':       'outerwear',
    'jackets_coats':       'outerwear',  # confirmed unmapped on first run → outerwear
    'outerwear':           'outerwear',
    'blazers_sport_coats': 'outerwear',
    'coats':               'outerwear',
    'cardigans':           'outerwear',
    'suiting':             'outerwear',  # confirmed unmapped on first run → outerwear
    # dress / full-body
    'dresses':             'dress',
    'rompers_jumpsuits':   'dress',
    'overalls':            'dress',
    # footwear (not present in this dataset but mapped for safety)
    'shoes':               'footwear',
    'boots':               'footwear',
    'sneakers':            'footwear',
    'sandals':             'footwear',
}

dfmm_path  = Path(DFMM_DIR)
dfmm_added = 0
dfmm_skipped_types = defaultdict(int)

if not dfmm_path.exists():
    print('[DFMM] dataset not found — skipping')
else:
    csv_path = next(
        (dfmm_path / f for f in ('labels_front.csv', 'labels_front.feather')
         if (dfmm_path / f).exists()),
        next(dfmm_path.rglob('labels_front.csv'), None)
    )
    img_dir = next(
        (p for p in [dfmm_path / 'selected_images', dfmm_path / 'images'] if p.exists()),
        next(dfmm_path.rglob('selected_images'), None)
    )

    if csv_path is None or img_dir is None:
        print(f'[DFMM] Could not find labels CSV or image dir — skipping')
    else:
        df_dfmm = pd.read_feather(csv_path) if csv_path.suffix == '.feather' else pd.read_csv(csv_path)
        print(f'[DFMM] {len(df_dfmm):,} rows  |  CSV: {csv_path.name}  |  images: {img_dir.name}/')

        pt_counts = df_dfmm['product_type'].value_counts()
        print(f'[DFMM] {len(pt_counts)} product_types:')
        for pt, cnt in pt_counts.items():
            key = str(pt).lower().replace(' ', '_').replace('/', '_')
            lbl = DFMM_PRODUCT_MAP.get(key)
            print(f'  {str(pt):<30} {cnt:>5}  →  {lbl or "UNMAPPED (skipped)"}')

        for _, row in df_dfmm.iterrows():
            pt  = str(row.get('product_type', '')).lower().replace(' ', '_').replace('/', '_')
            lbl = DFMM_PRODUCT_MAP.get(pt)
            if lbl is None:
                dfmm_skipped_types[pt] += 1
                continue
            fname = str(row.get('path', '')).strip()
            if not fname:
                continue
            src = img_dir / fname
            if not src.exists():
                src = img_dir / (Path(fname).stem + '.jpg')
            if not src.exists():
                continue
            records.append(make_row(src, {lbl}))
            dfmm_added += 1

        if dfmm_skipped_types:
            print(f'\n[DFMM] Still-unmapped types:')
            for pt, cnt in sorted(dfmm_skipped_types.items(), key=lambda x: -x[1]):
                print(f'  {pt}: {cnt}')
        else:
            print('[DFMM] All product_types mapped — zero skips.')

        print(f'[DFMM] added {dfmm_added:,} records')

print(f'After DeepFashion-MultiModal: {len(records):,} records')

In [ ]:
# ── Balance ────────────────────────────────────────────────────────────────
rng = random.Random(SEED)
rng.shuffle(records)
counts = defaultdict(int)
kept = []
for rec in records:
    present = [l for l in LABELS if rec[l] == 1]
    if not present: continue
    if any(counts[l] < MAX_PER_CLASS for l in present):
        kept.append(rec)
        for l in present: counts[l] += 1

print(f'Total after balancing: {len(kept):,}')
for l in LABELS:
    print(f'  {l:<12}: {counts[l]:,}')

In [ ]:
# ── Split + copy images (parallel) ────────────────────────────────────────
rng2 = random.Random(SEED)
rng2.shuffle(kept)
n = len(kept)
n_test = int(n * TEST_FRAC)
n_val  = int(n * VAL_FRAC)
splits_data = {
    'test':  kept[:n_test],
    'val':   kept[n_test:n_test+n_val],
    'train': kept[n_test+n_val:],
}
print('Split sizes:', {k: len(v) for k,v in splits_data.items()})

def copy_image(rec, dst_dir):
    src = Path(rec['filepath'])
    dst = dst_dir / src.name
    if not dst.exists(): shutil.copy2(src, dst)
    rec['filepath'] = str(dst)

for split_name, split_records in splits_data.items():
    dst_dir = Path(OUT_DIR) / 'images' / split_name
    dst_dir.mkdir(parents=True, exist_ok=True)
    with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
        futs = [ex.submit(copy_image, rec, dst_dir) for rec in split_records]
        for fut in tqdm(concurrent.futures.as_completed(futs), total=len(futs), desc=f'Copying {split_name}'):
            fut.result()

print('Images copied')

In [ ]:
# ── Write CSVs + stats ─────────────────────────────────────────────────────
for split_name, split_records in splits_data.items():
    out_path = f'{OUT_DIR}/labels_{split_name}.csv'
    pd.DataFrame(split_records).to_csv(out_path, index=False)
    print(f'Wrote {len(split_records):,} rows → {out_path}')

stats = {
    'total': len(kept),
    'splits': {k: len(v) for k,v in splits_data.items()},
    'label_counts': {
        sn: {l: sum(r[l] for r in recs) for l in LABELS}
        for sn, recs in splits_data.items()
    }
}
with open(f'{OUT_DIR}/stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print('\nFinal stats:')
print(json.dumps(stats, indent=2))
print('\nDone! Save output as dataset: rizzvision-clothing-dataset-v3')